# VLM-DENTAL Master Colab Workspace (Dataset & Data Gen)

This notebook contains the heavy dataset-downloading, trace generation, and YOLO training steps. SFT and GRPO training are in separate notebooks.

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**(Optional) Fresh Start Cleanup:**
Run this cell ONLY if you need to completely delete the VLM-DENTAL folder from your Google Drive to start over.

In [ ]:
# Uncomment the line below to delete the folder, then run the cell
# !rm -rf /content/drive/MyDrive/VLM-DENTAL

In [ ]:
import os

# Set this to True to save the 10GB dataset and code to Google Drive.
# Set this to False to download them to temporary Colab storage (faster, but lost when Colab disconnects).
# NOTE: Regardless of this setting, your generated traces and YOLO weights will ALWAYS be saved to Google Drive.
SAVE_DATASET_AND_CODE_TO_DRIVE = True

drive_path = "/content/drive/MyDrive/VLM-DENTAL"
colab_path = "/content/VLM-DENTAL"
work_dir = drive_path if SAVE_DATASET_AND_CODE_TO_DRIVE else colab_path




In [ ]:
if SAVE_DATASET_AND_CODE_TO_DRIVE:
    %cd /content/drive/MyDrive
else:
    %cd /content

import os
if not os.path.exists("VLM-DENTAL"):
    !git clone https://github.com/rezaxr14/VLM-DENTAL.git

%cd {work_dir}
!git pull

# Ensure output directories exist in Drive
os.makedirs(f"{drive_path}/data/traces", exist_ok=True)
os.makedirs(f"{drive_path}/data/models", exist_ok=True)

In [ ]:
# Install the project and all its requirements
!pip install -e .
!pip install python-dotenv pandas pillow google-generativeai anthropic huggingface_hub ultralytics

## 2. Configure API Keys

In [ ]:
import os

# Replace the placeholder below with your actual Gemini API keys (comma-separated for multiple)
os.environ['GEMINI_API_KEYS'] = 'YOUR_API_KEY_1,YOUR_API_KEY_2'

# Paste your Hugging Face token here directly for faster dataset downloads:
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'

print("API Keys are set! Note: Make sure you don't commit this notebook to github with your real keys exposed.")

## 3. Dataset Download & Cleanup
Run this to download the dataset if you haven't already. It will extract and structure it automatically.

In [ ]:
!python download_and_cleanup.py

In [ ]:
# Delete the unused partial datasets to save space
!rm -rf data/dentex/DENTEX/training_data/disease
!rm -rf data/dentex/DENTEX/training_data/quadrant
!rm -rf data/dentex/DENTEX/training_data/quadrant_enumeration

!rm -rf data/dentex/DENTEX/testing_data/disease
!rm -rf data/dentex/DENTEX/testing_data/quadrant
!rm -rf data/dentex/DENTEX/testing_data/quadrant_enumeration

# Delete the corrupted cache folder from any previous bugs (if it exists)
!rm -rf "C:\Users\rezax\dental_agent_cache"

## 4. Autonomous Trace Generation
Runs the daily trace generator and continuously saves progress to `train_cot_traces.jsonl`.

In [ ]:
!python scripts/run_daily_trace_generator.py --split train --output /content/drive/MyDrive/VLM-DENTAL/data/traces/train_cot_traces.jsonl

## 5. YOLO Grounding Tool Training
Convert COCO annotations to YOLO format, then train the model for 500 epochs.

In [ ]:
# Convert COCO annotations to YOLO format
!python scripts/prepare_yolo_dataset.py

In [ ]:
# Train YOLO for 500 epochs
# Your weights will automatically be saved to your Google Drive at: /content/drive/MyDrive/VLM-DENTAL/data/models/grounding_tool_heavy/weights/best.pt
!yolo train data=data/yolo_dentex/dataset.yaml model=yolov8m.pt epochs=500 imgsz=640 batch=16 device=0 project=/content/drive/MyDrive/VLM-DENTAL/data/models name=grounding_tool_heavy resume=True

## 6. YOLO Grounding Tool Training (X-Large Model)
Train a second, larger model (yolov8x) for potentially better performance.

In [ ]:
# Train larger YOLO model
# Your weights will automatically be saved to your Google Drive at: /content/drive/MyDrive/VLM-DENTAL/data/models/grounding_tool_xlarge/weights/best.pt
!yolo train data=data/yolo_dentex/dataset.yaml model=yolov8x.pt epochs=500 imgsz=640 batch=16 device=0 project=/content/drive/MyDrive/VLM-DENTAL/data/models name=grounding_tool_xlarge